# Gate B — collision cache vs twin (Kaggle T4×2)

Same SmolLM2 3:1 student as Gate A. One 20-minute transfer **without** the cache, then one **with** `--cache` (K=32, τ=0.5). A 2-step cache dry run comes first. Never P100. Never bf16.

ΔMSE is `(MSE_without − MSE_with) / MSE_without` per layer. Do not fill `docs/decisions/gate-B.md` from a crash.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/Caedral-ai/notrehybrid.git"
cwd = Path.cwd()

if (cwd / "setup_kaggle.sh").exists():
    root = cwd
elif (cwd / "notrehybrid" / "setup_kaggle.sh").exists():
    root = cwd / "notrehybrid"
else:
    !git clone --depth 1 {REPO} notrehybrid
    root = cwd / "notrehybrid"

os.chdir(root)
print("repo root:", root)
!bash setup_kaggle.sh

In [ ]:
WORK = "/kaggle/working"
TEACHER = f"{WORK}/teachers/SmolLM2-360M"
TAYLOR = f"{WORK}/checkpoints/gate-b/init-taylor"
CKPT = f"{WORK}/checkpoints/gate-b"
CFG = "configs/smollm2_360m/gate_a.yaml"
print(TEACHER, TAYLOR, CKPT)

In [ ]:
!python -m notre.convert.convert_smollm2 --hf HuggingFaceTB/SmolLM2-360M --out {TEACHER}
!python -m notre.convert.taylor_calibrate --cfg {CFG} --output {TAYLOR} --teacher {TEACHER} --hf-teacher HuggingFaceTB/SmolLM2-360M

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
TEACHER=/kaggle/working/teachers/SmolLM2-360M
TAYLOR=/kaggle/working/checkpoints/gate-b/init-taylor
CKPT=/kaggle/working/checkpoints/gate-b
CFG=configs/smollm2_360m/gate_a.yaml
python -m notre.convert.transfer --cfg "$CFG" --teacher "$TEACHER" --student-init "$TAYLOR" --ckpt-dir "$CKPT" --cache --slots 32 --tau 0.5 --minutes 20 --save-every 10 --keep-last 2

In [ ]:
from pathlib import Path
root = Path("/kaggle/working/checkpoints/gate-b")
for name in ("transfer/mse.csv", "transfer-cache/mse.csv", "transfer/mse_layers.csv", "transfer-cache/mse_layers.csv"):
    path = root / name
    print("==", name)
    print(path.read_text() if path.exists() else "missing")
